## ENERF images preparation. Use our provided COLMAP models, init point clouds and masks. 

In this notebook we show steps required to prepare ENeRF images for training.   
In paper we used two sequences:
- **Actor 1 (timesteps 710 - 780)**  
- **Actor 3 (timesteps 895-950)**


In our Zenodo repository we provide folders with following structures:
```
enerf_actors_1_3
│
├── actor1_710_780
│   ├── colmap_bg
│   ├── colmap_bg_undistorted
│   ├── masks_710_780_undistorted
│   └── merged_pcd.ply
│
└── actor3_895_950
    ├── colmap_bg
    ├── colmap_bg_undistorted
    ├── masks_895_950_undistorted
    └── merged_pcd.ply
```

We provide colmap reconstruction **on background images**, and you need to reuse this colmap model on dynamic images (with actors). Running colmap reconstruction on dynamic images directly (even on a single step) gives inaccurate estimations.

To get ENERF images you need to access the download link from: https://github.com/zju3dv/ENeRF/blob/master/docs/enerf_outdoor.md.  
Once you obtain the link, use ENERF folder for selected actor.  You will need COLMAP installed. This notebook shows steps for Actor1.


## 1. Set paths to our folder and DNA images for selected Actor, set timesteps.

In original ENERF data, for Actor1 `images` structure look like this:

```
images/
├── 00/ # camera idx
│   ├── 000000.jpg # timestep idx
│   ├── 000001.jpg
...
├── 17/                  
│   ├── 000000.jpg        
│   ├── 000001.jpg

```

In [ ]:
# === Set PATHS ======
LUMIMOTION_DIR = "path/to/enerf_actors_1_3/actor1_710_780"
IMAGE_DIR = "path/to/enerf/actor1/images" #path to enerf images with actor
# ===================

# === Set timesteps ===
min_ts = 710
max_ts = 780
# ===================

## 2. We train LumiMotion only on a subset of timesteps. 
Snippet below copies dynamic images for selected timesteps (for example 710-780). The output folder `images_710_780` will be structured as:

```
images_710_780/
├── 0000710/          # timestep idx
│   ├── 00.jpg        # camera idx
│   ├── 01.jpg
│   ├── ...
│   └── 17.jpg
└── ...
│
├── 0000780/
│   ├── 00.jpg
│   ├── 01.jpg
│   ├── ...
│   └── 17.jpg

```

In [ ]:
import os
import shutil
import re

source_root = IMAGE_DIR
dest_root = f"{LUMIMOTION_DIR}/images_{min_ts}_{max_ts}"
os.makedirs(dest_root, exist_ok=True)

# === Go through each camera directory
for cam_id in sorted(os.listdir(source_root)):
    cam_path = os.path.join(source_root, cam_id)
    if not os.path.isdir(cam_path):
        continue

    for filename in sorted(os.listdir(cam_path)):
        if not filename.lower().endswith(".jpg"):
            continue

        # Extract numeric timestamp from filename
        match = re.search(r"\d{6}", filename)
        if not match:
            continue
        ts = int(match.group())

        # Check timestamp range
        if min_ts <= ts <= max_ts:
            # Create target directory and copy file
            target_dir = os.path.join(dest_root, f"{ts:06d}")
            os.makedirs(target_dir, exist_ok=True)

            src_file = os.path.join(cam_path, filename)
            dst_file = os.path.join(target_dir, f"{cam_id}.jpg")

            shutil.copyfile(src_file, dst_file)


## 4. Undistort copied dynamic images with COLMAP undistortion model obtained for background.
The output folder: `images_710_780_undistorted` (should have the same structure as `images_710_780`)

In [ ]:
import os
import subprocess


IMAGES_COPIED_UNDISTORTED_PATH = f"{dest_root}_undistorted"
os.makedirs(IMAGES_COPIED_UNDISTORTED_PATH, exist_ok=True)

folders = sorted([f for f in os.listdir(dest_root) if os.path.isdir(os.path.join(dest_root, f))])

for folder in folders:
    image_path = os.path.join(dest_root, folder)
    output_path = os.path.join(IMAGES_COPIED_UNDISTORTED_PATH, folder)
    os.makedirs(output_path, exist_ok=True)

    print(f"[INFO] Undistorting: {folder}")

    subprocess.run([
        "colmap", "image_undistorter",
        "--image_path", image_path,
        "--input_path", f"{LUMIMOTION_DIR}/colmap_bg/sparse/0",
        "--output_path", output_path,
        "--output_type", "COLMAP"
    ], check=True)

    # === Cleanup: Remove 'sparse' and 'stereo' folders ===
    sparse_dir = os.path.join(output_path, "sparse")
    stereo_dir = os.path.join(output_path, "stereo")

    if os.path.exists(sparse_dir):
        shutil.rmtree(sparse_dir)
        print(f"🧹 Removed: {sparse_dir}")

    if os.path.exists(stereo_dir):
        shutil.rmtree(stereo_dir)
        print(f"🧹 Removed: {stereo_dir}")



## 5. Final dataset structure should look like this:
```
actor1_710_780/
├── colmap_bg_undistorted/
├── images_710_780_undistorted/
├── masks_710_780_undistorted/
└── merged_pcd.ply
```
`colmap_bg` and `images_710_780` folders can stay, but they won't be read.

We use path to this folder `actor1_710_780` in bash script.


## 6. Final notes on Stage2 optimization with masks. 
Please note, that provided scene masks are still quite large - large parts of wall will optimized in Stage2, which may deteriorate material estimation. 
To better focus on shadowed parts, after Stage1 you can remove some wall-gaussians (from areas with no shadow) from `point_cloud/iteration_35000/point_cloud.ply` in SuperSplat and replace the .ply file. We did it for the figures in our paper.